In [0]:
# ============================================================
# MyCredit-IQ | 01_bronze_loans_synthetic
# Synthetic Malaysian loan portfolio generator
# Target: 50,000 borrowers, ~7-10% default rate, realistic DSR
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

np.random.seed(42)
random.seed(42)

N = 50_000

In [0]:
# ============================================================
# SECTION 1: BORROWER PROFILE
# ============================================================

# ── Income brackets ──
income_bracket = np.random.choice(
    ["B40", "M40", "T20"],
    size=N,
    p=[0.40, 0.40, 0.20]
)

# ── Monthly income (RM) ──
def sample_income(bracket):
    if bracket == "B40":
        return np.clip(np.random.lognormal(mean=7.8, sigma=0.35), 1_200, 4_849)
    elif bracket == "M40":
        return np.clip(np.random.lognormal(mean=8.7, sigma=0.30), 4_850, 10_959)
    else:
        return np.clip(np.random.lognormal(mean=9.8, sigma=0.45), 10_960, 60_000)

monthly_income = np.array([sample_income(b) for b in income_bracket])
annual_income  = monthly_income * 12

# ── Employment sector ──
sectors = [
    "Government / Public Sector",
    "Financial Services",
    "Manufacturing",
    "Wholesale & Retail",
    "Construction",
    "Professional Services",
    "Education",
    "Healthcare",
    "Technology",
    "Agriculture"
]
sector_weights = [0.15, 0.10, 0.18, 0.14, 0.10, 0.08, 0.07, 0.06, 0.07, 0.05]
employment_sector = np.random.choice(sectors, size=N, p=sector_weights)

# ── Employment type ──
employment_type = np.random.choice(
    ["Permanent", "Contract", "Self-Employed", "Gig Worker"],
    size=N,
    p=[0.55, 0.20, 0.18, 0.07]
)

# ── Existing credit facilities ──
n_credit_facilities = np.random.poisson(lam=2.2, size=N).clip(0, 8).astype(int)

# ── Days past due history ──
dpd_history = np.random.choice(
    [0, 30, 60, 90, 120, 180],
    size=N,
    p=[0.72, 0.12, 0.07, 0.05, 0.03, 0.01]
)

In [0]:
# ============================================================
# SECTION 2: LOAN ATTRIBUTES
# Malaysian personal loans: max 10x monthly salary (BNM guideline)
# Hire purchase: vehicle financing, typically RM40k-RM150k
# Home loan: 3-5x annual income, 20-35 year tenure
# ============================================================

loan_type = np.random.choice(
    ["Personal Loan", "Hire Purchase", "Home Loan"],
    size=N,
    p=[0.40, 0.35, 0.25]
)

loan_amount = np.where(loan_type == "Personal Loan", monthly_income * np.random.uniform(2.0, 8.0, N), np.where(loan_type == "Hire Purchase", np.random.uniform(30_000, 180_000, N), annual_income * np.random.uniform(3.0, 5.0, N)))

# ── Loan tenure by type ──
def sample_tenure(lt):
    if lt == "Personal Loan":
        return np.random.choice([24, 36, 48, 60, 84])
    elif lt == "Hire Purchase":
        return np.random.choice([60, 72, 84, 96])
    else:
        return np.random.choice([180, 240, 300, 360])

loan_tenure_months = np.array([sample_tenure(lt) for lt in loan_type])

# ── Interest rate by loan type (post-BNM OPR environment) ──
def sample_rate(lt, bracket):
    base = {
        "Personal Loan":  np.random.uniform(0.060, 0.110),  # 6-11% flat
        "Hire Purchase":  np.random.uniform(0.025, 0.045),  # 2.5-4.5% flat
        "Home Loan":      np.random.uniform(0.035, 0.055),  # 3.5-5.5% reducing
    }[lt]
    # B40 borrowers face slightly higher rates (risk-based pricing)
    premium = 0.005 if bracket == "B40" else 0.0
    return base + premium

annual_interest_rate = np.array([sample_rate(lt, b) for lt, b in zip(loan_type, income_bracket)])

# ── Monthly instalment (reducing balance) ──
monthly_rate = annual_interest_rate / 12
monthly_instalment = np.where(monthly_rate > 0, (loan_amount * monthly_rate) / (1 - (1 + monthly_rate) ** (-loan_tenure_months)), loan_amount / loan_tenure_months)

# ── Debt Service Ratio ──
dsr = (monthly_instalment / monthly_income).clip(0, 1.5)

# Sanity check
print("── DSR Distribution ──")
print(pd.Series(dsr).describe(percentiles=[.25, .50, .70, .80, .90, .95]).round(3))
print(f"DSR > 60%: {(dsr > 0.60).mean():.1%}")
print(f"DSR > 80%: {(dsr > 0.80).mean():.1%}")

# ── Origination date ──
start_date  = datetime(2020, 1, 1)
days_range  = (datetime(2024, 12, 31) - start_date).days
origination_date = [start_date + timedelta(days=random.randint(0, days_range)) for _ in range(N)]
origination_ym   = [d.strftime("%Y-%m") for d in origination_date]
origination_year = [d.year for d in origination_date]

In [0]:
# ============================================================
# SECTION 3: DEFAULT LABEL
# ── Calibrated to ~7-10% NPL rate
# ============================================================

# Base rate
default_prob = np.full(N, 0.015)

# DSR stress
default_prob += (dsr > 0.60).astype(float) * 0.035
default_prob += (dsr > 0.80).astype(float) * 0.060

# DPD history
default_prob += (dpd_history == 30).astype(float)  * 0.08
default_prob += (dpd_history == 60).astype(float)  * 0.14
default_prob += (dpd_history >= 90).astype(float)  * 0.22

# Employment type
default_prob += np.where(employment_type == "Self-Employed", 0.025, 0.0)
default_prob += np.where(employment_type == "Gig Worker",    0.045, 0.0)

# Income bracket
default_prob += np.where(income_bracket == "B40", 0.018, 0.0)

# Loan type
default_prob += np.where(loan_type == "Personal Loan", 0.020, 0.0)

# Credit facility overload
default_prob += np.where(n_credit_facilities > 5, 0.025, 0.0)

# Origination during COVID recession (2020)
default_prob += np.where(np.array(origination_year) == 2020, 0.015, 0.0)

default_prob = default_prob.clip(0, 0.95)
is_default   = np.random.binomial(1, default_prob)

# ── Validation output ──
print("\n── Default Label Validation ──")
print(f"Overall default rate: {is_default.mean():.2%}")
print(f"Default rate — B40: {is_default[income_bracket == 'B40'].mean():.2%}")
print(f"Default rate — M40: {is_default[income_bracket == 'M40'].mean():.2%}")
print(f"Default rate — T20: {is_default[income_bracket == 'T20'].mean():.2%}")
print(f"Default rate — Gig Workers: {is_default[employment_type == 'Gig Worker'].mean():.2%}")
print(f"Default rate — Government: {is_default[employment_sector == 'Government / Public Sector'].mean():.2%}")
print(f"Default rate — DPD > 0: {is_default[dpd_history > 0].mean():.2%}")
print(f"Default rate — DSR > 80%: {is_default[dsr > 0.80].mean():.2%}")

In [0]:
# ============================================================
# SECTION 4: ASSEMBLE DATAFRAME
# ============================================================

df = pd.DataFrame({
    "borrower_id": [f"MY{str(i).zfill(6)}" for i in range(N)],
    "income_bracket": income_bracket,
    "monthly_income": monthly_income.round(2),
    "annual_income": annual_income.round(2),
    "employment_sector": employment_sector,
    "employment_type": employment_type,
    "n_credit_facilities": n_credit_facilities,
    "days_past_due_history": dpd_history,
    "loan_type": loan_type,
    "loan_amount": loan_amount.round(2),
    "loan_tenure_months": loan_tenure_months,
    "annual_interest_rate": annual_interest_rate.round(4),
    "monthly_instalment": monthly_instalment.round(2),
    "debt_service_ratio": dsr.round(4),
    "origination_date": origination_date,
    "origination_ym": origination_ym,
    "origination_year": origination_year,
    "is_default": is_default.astype(int),
})

# ── Final distribution checks ──
print("\n── Loan Type Distribution ──")
print(df["loan_type"].value_counts())

print("\n── Income Bracket Distribution ──")
print(df["income_bracket"].value_counts())

print("\n── Employment Type Distribution ──")
print(df["employment_type"].value_counts())

print("\n── Loan Amount by Type (RM) ──")
print(df.groupby("loan_type")["loan_amount"].describe().round(0))

print("\n── Monthly Instalment Summary (RM) ──")
print(df["monthly_instalment"].describe().round(2))

In [0]:
# ============================================================
# SECTION 5: WRITE TO DELTA (BRONZE LAYER)
# ============================================================

sdf = spark.createDataFrame(df)

(sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("my_living_index.credit.loans_bronze"))

print(f"\nWritten {N:,} rows → my_living_index.credit.loans_bronze")
sdf.printSchema()